In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q rdkit xgboost chemprop

import pandas as pd, numpy as np, subprocess, os
base = "/content/drive/MyDrive/NIR_Thesis"
clean = pd.read_csv(f"{base}/Data/my_data_clean.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
def make_split(df, seed):
    groups = df.groupby("scaffold").size().sample(frac=1, random_state=seed)
    big = groups[groups > len(df) * 0.005].index.tolist()
    small = [g for g in groups.index if g not in set(big)]
    split, count, n = {}, 0, len(df)
    for g in big + small:
        split[g] = "train" if count < 0.8 * n else ("val" if count < 0.9 * n else "test")
        count += groups[g]
    return df.scaffold.map(split)

SEEDS = [42, 0, 1, 2, 3]
for s in SEEDS:
    clean[f"split_{s}"] = make_split(clean, s)

assert (clean["split_42"] == clean["split"]).all()   # seed 42 must reproduce your original split
for s in SEEDS:
    t = clean[clean[f"split_{s}"] == "test"]
    print(f"seed {s}: test={len(t)}  >=650 nm: {(t.emi_nm >= 650).sum()}  >=700 nm: {(t.emi_nm >= 700).sum()}")

seed 42: test=1779  >=650 nm: 143  >=700 nm: 77
seed 0: test=1773  >=650 nm: 147  >=700 nm: 54
seed 1: test=1782  >=650 nm: 74  >=700 nm: 32
seed 2: test=1785  >=650 nm: 176  >=700 nm: 83
seed 3: test=1775  >=650 nm: 119  >=700 nm: 58


In [4]:
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
RDLogger.DisableLog("rdApp.*")

morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
solv_morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=256)

def conjugation_features(m):
    n_bonds, n_atoms = max(m.GetNumBonds(), 1), max(m.GetNumHeavyAtoms(), 1)
    conj = sum(b.GetIsConjugated() for b in m.GetBonds())
    sp2 = sum(a.GetHybridization() in (Chem.HybridizationType.SP2, Chem.HybridizationType.SP) for a in m.GetAtoms())
    arom = sum(a.GetIsAromatic() for a in m.GetAtoms())
    return [conj, conj / n_bonds, sp2 / n_atoms, arom / n_atoms, Chem.rdMolDescriptors.CalcNumAromaticRings(m)]

def chrom_features(smi):
    m = Chem.MolFromSmiles(smi)
    return np.concatenate([morgan.GetCountFingerprintAsNumPy(m),
                           list(Descriptors.CalcMolDescriptors(m).values()),
                           conjugation_features(m)]).astype(float)

def solv_features(smi):
    m = Chem.MolFromSmiles(smi)
    d = [Descriptors.MolLogP(m), Descriptors.TPSA(m), Descriptors.MolWt(m),
         Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m), Chem.rdMolDescriptors.CalcNumAromaticRings(m)]
    return np.concatenate([solv_morgan.GetCountFingerprintAsNumPy(m), d]).astype(float)

chrom_cache = {s: chrom_features(s) for s in clean.chrom.unique()}
solv_cache = {s: solv_features(s) for s in clean.solv.unique()}
X = np.array([np.concatenate([chrom_cache[c], solv_cache[s]]) for c, s in zip(clean.chrom, clean.solv)])
X = np.clip(np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0), -1e6, 1e6)

np.save(f"{base}/Data/X_features.npy", X)
print(X.shape)

(17850, 1508)


In [5]:
from sklearn.metrics import mean_absolute_error, r2_score

BINS = {"overall": (0, 2000), "<500": (0, 500), "500-650": (500, 650),
        ">=650": (650, 2000), ">=700": (700, 2000)}

def evaluate(true, pred):
    r = {"R2": r2_score(true, pred)}
    for name, (lo, hi) in BINS.items():
        m = (true >= lo) & (true < hi)
        r[f"n {name}"] = int(m.sum())
        r[f"MAE {name}"] = mean_absolute_error(true[m], pred[m])
        r[f"bias {name}"] = np.mean(pred[m] - true[m])
    return r

In [6]:
import xgboost as xgb

y = clean.emi_nm.values
results, all_preds = [], []

for s in SEEDS:
    sp = clean[f"split_{s}"].values
    tr, va, te = sp == "train", sp == "val", sp == "test"
    true = y[te]

    # --- XGBoost (same settings as notebook 02) ---
    m = xgb.XGBRegressor(n_estimators=3000, learning_rate=0.03, max_depth=8, subsample=0.8,
                         colsample_bytree=0.4, min_child_weight=2, tree_method="hist",
                         early_stopping_rounds=150, random_state=s, device="cuda")
    m.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=False)
    p_xgb = m.predict(X[te])
    results.append({"seed": s, "model": "XGBoost", **evaluate(true, p_xgb)})

    # --- D-MPNN (same settings as notebook 03) ---
    for part, mask in [("train", tr), ("val", va), ("test", te)]:
        clean[mask][["chrom", "solv", "emi_eV"]].to_csv(f"/content/cp_{part}.csv", index=False)
    mdir = f"/content/dmpnn_seed{s}"
    cmd = (f"chemprop train --data-path /content/cp_train.csv /content/cp_val.csv /content/cp_test.csv "
           f"--smiles-columns chrom solv --target-columns emi_eV --task-type regression "
           f"--epochs 40 --patience 10 --batch-size 64 --message-hidden-dim 300 --depth 4 "
           f"--ffn-num-layers 2 --pytorch-seed {s} --save-dir {mdir}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise RuntimeError(f"training failed for seed {s}")
    r = subprocess.run(f"chemprop predict --test-path /content/cp_test.csv --smiles-columns chrom solv "
                       f"--model-paths {mdir} --preds-path /content/preds_{s}.csv",
                       shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise RuntimeError(f"prediction failed for seed {s}")
    p_gnn = 1239.84 / pd.read_csv(f"/content/preds_{s}.csv").emi_eV.values
    results.append({"seed": s, "model": "D-MPNN", **evaluate(true, p_gnn)})

    # --- save after every seed, so a disconnect doesn't lose everything ---
    all_preds.append(pd.DataFrame({"seed": s, "chrom": clean.chrom[te].values, "solv": clean.solv[te].values,
                                   "true_nm": true, "xgb_nm": p_xgb, "dmpnn_nm": p_gnn}))
    pd.DataFrame(results).to_csv(f"{base}/results/multiseed_results.csv", index=False)
    pd.concat(all_preds).to_csv(f"{base}/results/multiseed_predictions.csv", index=False)
    print(f"seed {s} done | XGB MAE {results[-2]['MAE overall']:.1f}  D-MPNN MAE {results[-1]['MAE overall']:.1f}"
          f" | >=700: XGB {results[-2]['MAE >=700']:.1f}  D-MPNN {results[-1]['MAE >=700']:.1f}")

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [11:49:38] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


seed 42 done | XGB MAE 32.7  D-MPNN MAE 32.3 | >=700: XGB 86.4  D-MPNN 66.7
seed 0 done | XGB MAE 38.9  D-MPNN MAE 30.7 | >=700: XGB 83.6  D-MPNN 44.6
seed 1 done | XGB MAE 30.6  D-MPNN MAE 30.6 | >=700: XGB 66.4  D-MPNN 63.5
seed 2 done | XGB MAE 39.4  D-MPNN MAE 37.8 | >=700: XGB 73.1  D-MPNN 57.9
seed 3 done | XGB MAE 32.5  D-MPNN MAE 34.7 | >=700: XGB 37.8  D-MPNN 24.7


In [7]:
df = pd.DataFrame(results)
cols = ["MAE overall", "R2", "MAE <500", "MAE 500-650", "MAE >=650", "MAE >=700", "bias >=700"]
summary = df.groupby("model")[cols].agg(["mean", "std"]).round(2)
print(summary)
summary.to_csv(f"{base}/results/multiseed_summary.csv")

# pool every split's NIR test predictions together
allp = pd.concat(all_preds)
for lo in [650, 700]:
    nir = allp[allp.true_nm >= lo]
    print(f"pooled >= {lo} nm (n={len(nir)}): XGB MAE {mean_absolute_error(nir.true_nm, nir.xgb_nm):.1f}  "
          f"D-MPNN MAE {mean_absolute_error(nir.true_nm, nir.dmpnn_nm):.1f}")

        MAE overall          R2       MAE <500       MAE 500-650        \
               mean   std  mean   std     mean   std        mean   std   
model                                                                    
D-MPNN        33.21  3.04  0.74  0.06    27.14  1.59       39.08  3.84   
XGBoost       34.80  4.07  0.73  0.04    29.94  3.52       37.14  3.02   

        MAE >=650        MAE >=700        bias >=700         
             mean    std      mean    std       mean    std  
model                                                        
D-MPNN      45.15   9.42     51.49  17.22     -33.58  22.78  
XGBoost     57.11  13.03     69.46  19.45     -67.38  20.35  
pooled >= 650 nm (n=659): XGB MAE 59.0  D-MPNN MAE 45.8
pooled >= 700 nm (n=304): XGB MAE 70.9  D-MPNN MAE 52.0


In [8]:
from scipy import stats
df = pd.DataFrame(results)
for col in ["MAE overall", "MAE >=650", "MAE >=700"]:
    x = df[df.model == "XGBoost"].set_index("seed")[col]
    g = df[df.model == "D-MPNN"].set_index("seed")[col]
    diff = x - g
    t, p = stats.ttest_rel(x, g)
    print(f"{col}: D-MPNN better by {diff.mean():.1f} ± {diff.std():.1f} nm, "
          f"wins {int((diff > 0).sum())}/5, paired t-test p = {p:.3f}")

MAE overall: D-MPNN better by 1.6 ± 3.9 nm, wins 3/5, paired t-test p = 0.417
MAE >=650: D-MPNN better by 12.0 ± 10.9 nm, wins 5/5, paired t-test p = 0.070
MAE >=700: D-MPNN better by 18.0 ± 13.2 nm, wins 5/5, paired t-test p = 0.039


In [9]:
allp = pd.read_csv(f"{base}/results/multiseed_predictions.csv")
allp["avg_nm"] = (allp.xgb_nm + allp.dmpnn_nm) / 2
nir = allp[allp.true_nm >= 700]
print("pooled >=700 MAE  XGB:", round(mean_absolute_error(nir.true_nm, nir.xgb_nm), 1),
      " D-MPNN:", round(mean_absolute_error(nir.true_nm, nir.dmpnn_nm), 1),
      " average:", round(mean_absolute_error(nir.true_nm, nir.avg_nm), 1))

pooled >=700 MAE  XGB: 70.9  D-MPNN: 52.0  average: 55.8
